# 07 — Regularization + Cross-Validation + Model Comparison
## Goal
Compare:
- Linear Regression
- Ridge (L2)
- Lasso (L1)
- ElasticNet (L1 + L2)

Regularization adds a penalty to discourage overly large coefficients.

### Why?
With many one-hot encoded features and correlated variables, a model can become unstable or overfit. Regularization gives us a controlled way to constrain coefficients.

We use **GridSearchCV** on the training data only. The test set remains untouched until final evaluation.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../outputs/cleaned/zomato_model_dataset.csv")

target = "Time_taken (min)"
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

def make_pipeline(model):
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

models = {
    "Linear Regression": make_pipeline(LinearRegression()),
    "Ridge": make_pipeline(Ridge()),
    "Lasso": make_pipeline(Lasso(max_iter=20000)),
    "ElasticNet": make_pipeline(ElasticNet(max_iter=20000))
}

param_grids = {
    "Ridge": {"model__alpha": [0.01, 0.1, 1, 10, 100]},
    "Lasso": {"model__alpha": [0.001, 0.01, 0.1, 1]},
    "ElasticNet": {
        "model__alpha": [0.001, 0.01, 0.1, 1],
        "model__l1_ratio": [0.1, 0.5, 0.9]
    }
}

results = []

# Plain Linear Regression
models["Linear Regression"].fit(X_train, y_train)
pred = models["Linear Regression"].predict(X_test)
results.append({
    "Model": "Linear Regression",
    "MAE": mean_absolute_error(y_test, pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
    "R2": r2_score(y_test, pred)
})

best_models = {"Linear Regression": models["Linear Regression"]}

# Tune regularized models using training data only.
for name in ["Ridge", "Lasso", "ElasticNet"]:
    search = GridSearchCV(
        estimator=models[name],
        param_grid=param_grids[name],
        scoring="neg_root_mean_squared_error",
        cv=3,
        n_jobs=-1
    )
    search.fit(X_train, y_train)

    best_models[name] = search.best_estimator_
    pred = search.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
        "Best Params": search.best_params_
    })

comparison = pd.DataFrame(results).sort_values("RMSE")
comparison.round(3)

,Model,MAE,RMSE,R2,Best Params
3,ElasticNet,4.816,6.057,0.584,"{'model__alpha': 0.001, 'model__l1_ratio': 0.9}"
2,Lasso,4.816,6.058,0.584,{'model__alpha': 0.001}
1,Ridge,4.817,6.058,0.584,{'model__alpha': 1}
0,Linear Regression,4.817,6.059,0.584,NaN


In [2]:
# Train-vs-test comparison for the final candidates.
generalization = []

for name, model in best_models.items():
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    generalization.append({
        "Model": name,
        "Train_RMSE": np.sqrt(mean_squared_error(y_train, train_pred)),
        "Test_RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
        "Train_R2": r2_score(y_train, train_pred),
        "Test_R2": r2_score(y_test, test_pred)
    })

pd.DataFrame(generalization).sort_values("Test_RMSE").round(3)

,Model,Train_RMSE,Test_RMSE,Train_R2,Test_R2
3,ElasticNet,6.052,6.057,0.584,0.584
2,Lasso,6.052,6.058,0.584,0.584
1,Ridge,6.052,6.058,0.584,0.584
0,Linear Regression,6.052,6.059,0.584,0.584


## Reading the comparison

**RMSE:** lower is better. It punishes large mistakes more than MAE.

**MAE:** lower is better and easy to interpret as average minutes of absolute error.

**R²:** higher is better. It measures improvement over the mean-prediction baseline.

**Regularization:** Ridge shrinks coefficients toward zero; Lasso can shrink some coefficients exactly to zero; ElasticNet combines both penalties.

A regularized model is not automatically better. We choose based on validation/test evidence.

In [3]:
# Save comparison table.
from pathlib import Path
Path("../outputs/model_results").mkdir(parents=True, exist_ok=True)
comparison.to_csv("../outputs/model_results/regression_comparison.csv", index=False)

display(comparison.sort_values("RMSE").round(3))

,Model,MAE,RMSE,R2,Best Params
3,ElasticNet,4.816,6.057,0.584,"{'model__alpha': 0.001, 'model__l1_ratio': 0.9}"
2,Lasso,4.816,6.058,0.584,{'model__alpha': 0.001}
1,Ridge,4.817,6.058,0.584,{'model__alpha': 1}
0,Linear Regression,4.817,6.059,0.584,NaN
